In [1]:
import pandas as pd
import numpy as np

### **UCDP**
---

UCDP Georeferenced Event Dataset (GED) Global version 25.1 spanning from 1989-01 to 2024-12

In [2]:
df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv', low_memory=False)
#filter the conflicts that only sign agreements
df_ucdp = df_ucdp[df_ucdp['type_of_violence']==1].reset_index(drop=True).copy()
relevant_columns = ['country','conflict_new_id', 'date_start','date_end','best', 'dyad_new_id', 'type_of_violence', 'region', 'conflict_name', 'side_a', 'side_b']
df_ucdp = df_ucdp[relevant_columns].sort_values(['country', 'conflict_new_id', 'date_start'])

#join with isocode and add isocode
isocodes = pd.read_csv('../../data/input/isocodes/isocodes_appended.csv', sep = ';')
df_ucdp = df_ucdp.merge(isocodes[['name', 'alpha_3']], left_on='country', right_on='name', how='left').rename(columns = {'alpha_3':'isocode'})
df_ucdp

,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,region,conflict_name,side_a,side_b,name,isocode
0,Afghanistan,259,2017-07-31 00:00:00.000,2017-07-31 00:00:00.000,6,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
1,Afghanistan,259,2021-08-26 00:00:00.000,2021-08-26 00:00:00.000,183,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
2,Afghanistan,259,2021-08-28 00:00:00.000,2021-08-28 00:00:00.000,2,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
3,Afghanistan,259,2021-08-29 00:00:00.000,2021-08-29 00:00:00.000,10,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
4,Afghanistan,333,1989-01-01 00:00:00.000,1989-01-01 00:00:00.000,4,727,1,Asia,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
271326,Yemen (North Yemen),16099,2024-05-30 00:00:00.000,2024-05-30 00:00:00.000,14,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM
271327,Yemen (North Yemen),16099,2024-06-13 00:00:00.000,2024-06-13 00:00:00.000,5,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM
271328,Yemen (North Yemen),16099,2024-09-10 00:00:00.000,2024-09-10 00:00:00.000,2,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM
271329,Yemen (North Yemen),16099,2024-11-12 00:00:00.000,2024-11-12 00:00:00.000,10,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM


In [3]:
df_ucdp.groupby('conflict_new_id')['date_start'].nunique()

conflict_new_id
205        67
209      1821
218       759
220         1
221       654
         ... 
16038      34
16069     200
16099      18
16292       4
16379       1
Name: date_start, Length: 201, dtype: int64

In [14]:
df_ucdp["date_start"] = pd.to_datetime(df_ucdp["date_start"]).dt.to_period("M").dt.to_timestamp()
df_ucdp["date_end"]   = pd.to_datetime(df_ucdp["date_end"]).dt.to_period("M").dt.to_timestamp()

# Number of months in each row (inclusive)
n_months = (
    (df_ucdp["date_end"].dt.to_period("M") - df_ucdp["date_start"].dt.to_period("M"))
    .apply(lambda x: x.n + 1)
)

# Repeat rows for each month
df_expanded = df_ucdp.loc[df_ucdp.index.repeat(n_months)].copy()

month_offsets = df_expanded.groupby(level=0).cumcount()

df_expanded["date"] = (
    df_expanded["date_start"].dt.to_period("M")
    .add(month_offsets.values)
    .dt.to_timestamp()
)

df_expanded["best"] = df_expanded["best"] / n_months.repeat(n_months).values
df_expanded

,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,region,conflict_name,side_a,side_b,name,isocode,date
0,Afghanistan,259,2017-07-01,2017-07-01,6.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,2017-07-01
1,Afghanistan,259,2021-08-01,2021-08-01,183.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,2021-08-01
2,Afghanistan,259,2021-08-01,2021-08-01,2.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,2021-08-01
3,Afghanistan,259,2021-08-01,2021-08-01,10.0,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,2021-08-01
4,Afghanistan,333,1989-01-01,1989-01-01,4.0,727,1,Asia,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG,1989-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271326,Yemen (North Yemen),16099,2024-05-01,2024-05-01,14.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,2024-05-01
271327,Yemen (North Yemen),16099,2024-06-01,2024-06-01,5.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,2024-06-01
271328,Yemen (North Yemen),16099,2024-09-01,2024-09-01,2.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,2024-09-01
271329,Yemen (North Yemen),16099,2024-11-01,2024-11-01,10.0,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,2024-11-01


In [15]:
#2. Aggregate to conflict-month level and sum the 'best' values without distinguishing countries
#-----------------------------------------------------------------------------------------------
df_conflict_month = (
    df_expanded
    .groupby(['conflict_new_id','date'], as_index=False)
    .agg(
        best=('best','sum'),
        n_events=('dyad_new_id','count'),
        dyad_id = ('dyad_new_id', 'unique'),
        isocode = ('isocode','first'),
        country = ('country','first'),
        region = ('region','first'),
        n_isocode=('isocode','nunique'),
        isocode_array=('isocode', 'unique'),
        countries_array=('country','unique'),
        type_of_violence=('type_of_violence','min')
    )
)
df_conflict_month

,conflict_new_id,date,best,n_events,dyad_id,isocode,country,region,n_isocode,isocode_array,countries_array,type_of_violence
0,205,1990-04-01,0.0,1,[406],IRN,Iran,Middle East,1,[IRN],[Iran],1
1,205,1990-05-01,1.0,1,[406],IRN,Iran,Middle East,1,[IRN],[Iran],1
2,205,1990-06-01,23.0,6,[406],IRN,Iran,Middle East,1,[IRN],[Iran],1
3,205,1990-07-01,7.0,5,[406],IRN,Iran,Middle East,1,[IRN],[Iran],1
4,205,1990-08-01,0.0,1,[406],IRN,Iran,Middle East,1,[IRN],[Iran],1
...,...,...,...,...,...,...,...,...,...,...,...,...
16048,16292,2018-07-01,3.0,1,[15661],TJK,Tajikistan,Asia,1,[TJK],[Tajikistan],1
16049,16292,2018-11-01,49.0,1,[15661],TJK,Tajikistan,Asia,1,[TJK],[Tajikistan],1
16050,16292,2019-05-01,32.0,1,[15661],TJK,Tajikistan,Asia,1,[TJK],[Tajikistan],1
16051,16292,2019-11-01,17.0,1,[15661],TJK,Tajikistan,Asia,1,[TJK],[Tajikistan],1


Assign main country to each conflict_id as the country with the highest number of fatalities in that conflict.

In [5]:
# dominant country: the one with most events that month
main_isocode = (
    df_expanded.groupby(['conflict_new_id','isocode', 'country', 'region'])
    .agg(sum_best=('best','sum'))
    .reset_index()
    .sort_values(['conflict_new_id','sum_best'], ascending=[True,False])
    .drop_duplicates(['conflict_new_id'])
)
df_conflict_month = df_conflict_month.merge(
    main_isocode[['conflict_new_id','isocode', 'country', 'region']],
    on=['conflict_new_id'],
    how='left'
)
#drop the columns that should be replaced by main_isocode
df_conflict_month = df_conflict_month.drop(columns=['isocode_x','country_x', 'region_x'])
df_conflict_month = df_conflict_month.rename(columns={'isocode_y':'isocode', 'country_y':'country', 'region_y':'region',
                                                      'conflict_new_id':'conflict_id', 'date':'year_mo'})
df_conflict_month['best'] = df_conflict_month['best'].fillna(0.0)
df_conflict_month['log_best'] = np.log1p(df_conflict_month['best']) # log(1 + x) to handle 0 values
df_conflict_month

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,country,region,log_best
0,205,1990-04-01,0.0,1,[406],1,[IRN],[Iran],1,IRN,Iran,Middle East,0.000000
1,205,1990-05-01,1.0,1,[406],1,[IRN],[Iran],1,IRN,Iran,Middle East,0.693147
2,205,1990-06-01,23.0,6,[406],1,[IRN],[Iran],1,IRN,Iran,Middle East,3.178054
3,205,1990-07-01,7.0,5,[406],1,[IRN],[Iran],1,IRN,Iran,Middle East,2.079442
4,205,1990-08-01,0.0,1,[406],1,[IRN],[Iran],1,IRN,Iran,Middle East,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16048,16292,2018-07-01,3.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,Tajikistan,Asia,1.386294
16049,16292,2018-11-01,49.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,Tajikistan,Asia,3.912023
16050,16292,2019-05-01,32.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,Tajikistan,Asia,3.496508
16051,16292,2019-11-01,17.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,Tajikistan,Asia,2.890372


In [19]:
df_conflict_month.loc[df_conflict_month['conflict_id']==309]

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,country,region,log_best
5113,309,1989-01-01,370.0,2,[663],1,[SDN],[Sudan],1,SDN,Sudan,Africa,5.916202
5114,309,1989-02-01,302.0,3,[663],1,[SDN],[Sudan],1,SDN,Sudan,Africa,5.713733
5115,309,1989-03-01,372.0,4,[663],1,[SDN],[Sudan],1,SDN,Sudan,Africa,5.921578
5116,309,1989-04-01,560.0,5,[663],1,[SDN],[Sudan],1,SDN,Sudan,Africa,6.329721
5117,309,1989-05-01,270.0,1,[663],1,[SDN],[Sudan],1,SDN,Sudan,Africa,5.602119
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5448,309,2024-08-01,315.0,43,[17497],2,"[SSD, SDN]","[South Sudan, Sudan]",1,SDN,Sudan,Africa,5.755742
5449,309,2024-09-01,555.0,53,[17497],1,[SDN],[Sudan],1,SDN,Sudan,Africa,6.320768
5450,309,2024-10-01,658.0,50,[17497],1,[SDN],[Sudan],1,SDN,Sudan,Africa,6.490724
5451,309,2024-11-01,439.0,43,[17497],1,[SDN],[Sudan],1,SDN,Sudan,Africa,6.086775


### **Agreements**
---


In [ ]:
df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv', low_memory=False)
df_pax.columns = df_pax.columns.str.lower()

df_pax = df_pax.rename(columns={"ucdpcon": "conflict_id", "loc1iso": "isocode"})

df_pax["dat"] = pd.to_datetime(df_pax["dat"])
df_pax["year_mo"] = df_pax["dat"].dt.to_period("M").dt.to_timestamp()

# Replace SSD -> SDN before July 2011
mask = (df_pax["isocode"] == "SSD") & (
    (df_pax["dat"].dt.year < 2011) |
    ((df_pax["dat"].dt.year == 2011) & (df_pax["dat"].dt.month <= 6))
)
df_pax.loc[mask, "isocode"] = "SDN"

df_pax = df_pax.dropna(subset=['conflict_id', 'year_mo'])
df_pax["conflict_id"] = df_pax["conflict_id"].astype(int)

#Treatment indicators
df_pax["agreement"] = 1
df_pax["subs_agreement"] = (df_pax["stage"].str.lower() == "subpar").astype(int)
df_pax["comp_agreement"] = (df_pax["stage"].str.lower() == "subcomp").astype(int)
df_pax["cea_agreement"] = (df_pax["stage"].str.lower() == "cea").astype(int)
df_pax["cea_ceamix_agreement"] = (df_pax["stagesub"].str.lower() == "ceamix").astype(int)
df_pax["cea_ceas_agreement"] = (df_pax["stagesub"].str.lower() == "ceas").astype(int)
df_pax["cea_rel_agreement"] = (df_pax["stagesub"].str.lower() == "rel").astype(int)

#define variables to test clusterd Heterogenous TE
features_cluster_columns = ['hriimon', 'ime', 'ddrprog', 'imun', 'tjrep', 'ppsaut', 'tjmis', 
                            'protgrp', 'ppsvet', 'hriibod', 'tpsloc', 'terps', 'pol', 'epsres', 
                            'impk', 'epsoth', 'polnewtemp', 'protciv', 'hrtrinc', 'ssrddr', 
                            'ceprov', 'ppsint', 'ddrdemil', 'polpar', 'polps', 'hrbor', 'hrdem',
                            'medlog', 'tjvet', 'medsubs', 'epsfis', 'tpssub', 'tpsoth', 'eleccomm', 
                            'eps', 'ssrpol', 'medgov', 'imref', 'tpsaut', 'med', 'hrii', 'cegen', 'ssrgua',
                            'ssrarm', 'tjcou', 'ppsoro', 'tjamban', 'tjvic', 'tjmech', 'polnewind', 'mpsme', 
                            'mpsjt', 'imoth', 'ppsothpr', 'ppsex', 'hrmob', 'ele', 'hrni', 'civso', 'pubad', 
                            'juscr', 'mps', 'prot', 'mpspro']

df_agreements = (
    df_pax
    .groupby(["conflict_id", "year_mo"], as_index=False)
    .agg(
        agreement=("agreement", "max"),  
        comp_agreement=("comp_agreement", "max"),  
        subs_agreement=("subs_agreement", "max"),
        cea_agreement = ('cea_agreement', "max"), 
        cea_ceamix_agreement = ('cea_ceamix_agreement', "max"),
        cea_ceas_agreement = ('cea_ceas_agreement', "max"),
        cea_rel_agreement = ('cea_rel_agreement', "max"),
        ce=('ce','max'),
        **{var: (var, "max") for var in features_cluster_columns}
    )
)
df_agreements

,conflict_id,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,...,ppsex,hrmob,ele,hrni,civso,pubad,juscr,mps,prot,mpspro
0,209,1992-09-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,209,1994-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,209,1995-02-01,1,0,1,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
3,209,1995-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,209,1996-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1313,169170,1995-10-01,1,0,1,0,0,0,0,3,...,0,0,1,0,0,0,1,0,1,0
1314,169170,2000-12-01,1,0,0,0,0,0,0,2,...,0,0,0,0,1,0,0,0,0,0
1315,169170,2002-08-01,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1316,169170,2002-10-01,1,0,0,0,0,0,0,2,...,0,0,0,0,1,0,0,0,1,0


In [7]:
df_pax = df_pax.drop_duplicates(subset=['conflict_id', 'year_mo'])
relevant_columns = [i for i in df_pax.columns if not i.startswith('g') and i not in ['agreement', 'comp_agreement', 'subs_agreement', 
                                                                                     'cea_agreement', 'cea_ceas_agreement', 
                                                                                     'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns]
df_pax = df_pax[relevant_columns]
df_agreements = df_agreements.merge(
    df_pax,
    on=['conflict_id', 'year_mo'],
    how='left'    
)

In [8]:
df_panel = (
    df_conflict_month
    .merge(df_agreements, on=["conflict_id", "year_mo"], how="left", suffixes=("", "_agr"))
    .drop(columns=["isocode_agr"])
)

df_panel[["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns] = df_panel[
    ["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns
].fillna(0).astype(int)

df_panel['isocode_num'] = df_panel['isocode'].astype('category').cat.codes + 1
df_panel['region_num'] = df_panel['region'].astype('category').cat.codes + 1
df_panel['start_date'] = df_panel.groupby('conflict_id')['year_mo'].transform('min')
df_panel['end_date'] = df_panel.groupby('conflict_id')['year_mo'].transform('max')

df_panel

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,tjjaic,tjprire,tjrsym,tjrma,tjnr,imsrc,isocode_num,region_num,start_date,end_date
0,205,1990-04-01,0.0,1,[406],1,[IRN],[Iran],1,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,31,5,1990-04-01,2022-11-01
1,205,1990-05-01,1.0,1,[406],1,[IRN],[Iran],1,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,31,5,1990-04-01,2022-11-01
2,205,1990-06-01,23.0,6,[406],1,[IRN],[Iran],1,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,31,5,1990-04-01,2022-11-01
3,205,1990-07-01,7.0,5,[406],1,[IRN],[Iran],1,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,31,5,1990-04-01,2022-11-01
4,205,1990-08-01,0.0,1,[406],1,[IRN],[Iran],1,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,31,5,1990-04-01,2022-11-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16048,16292,2018-07-01,3.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,...,NaN,NaN,NaN,NaN,NaN,NaN,76,3,2018-07-01,2019-11-01
16049,16292,2018-11-01,49.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,...,NaN,NaN,NaN,NaN,NaN,NaN,76,3,2018-07-01,2019-11-01
16050,16292,2019-05-01,32.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,...,NaN,NaN,NaN,NaN,NaN,NaN,76,3,2018-07-01,2019-11-01
16051,16292,2019-11-01,17.0,1,[15661],1,[TJK],[Tajikistan],1,TJK,...,NaN,NaN,NaN,NaN,NaN,NaN,76,3,2018-07-01,2019-11-01


In [9]:
df_panel.loc[df_panel['agreement'] == 1]['conflict_id'].nunique()

71

## **Expand the panel**

From 1989-01 to 2024-12 monthly, but keeping the start_date and end_date of each conflict as in the original UCDP conflict dataset, to calculate conflict age and duration.


In [32]:
#4. Fill missing months with 0 fatalities to expand to a balanced panel
#-----------------------------------------------------------------------------------------------

global_start = df_ucdp.date_start.min()
global_end   = df_ucdp.date_end.max()

full_conflict_month = []

for conflict_id in df_panel['conflict_id'].unique():
    months = pd.date_range(global_start, global_end, freq='MS')
    tmp = pd.DataFrame({'conflict_id': conflict_id, 'year_mo': months})
    full_conflict_month.append(tmp)

full_panel = pd.concat(full_conflict_month, ignore_index=True)

# merge to fill missing months with 0 fatalities
df_panel = full_panel.merge(df_panel, on=['conflict_id','year_mo'], how='left')
df_panel = df_panel.sort_values(['conflict_id', 'year_mo']).reset_index(drop=True)
df_panel['real_observation'] = np.where(df_panel['best'].notna(), 1, 0)


# Fill numeric / indicator columns
columns_to_fill = ['best', 'agreement', 'comp_agreement','subs_agreement', 
                   'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement',
                   'cea_rel_agreement','n_events','n_isocode', 'ce'] + features_cluster_columns

df_panel[columns_to_fill] = df_panel[columns_to_fill].fillna(0)
df_panel['log_best'] = np.log1p(df_panel['best'])

# Fill categorical/meta columns within conflict
columns_to_ffill_bfill = ['isocode', 'country', 'isocode_num', 'region','region_num', 'isocode_array', 'countries_array', 'type_of_violence', 'start_date', 'end_date']
df_panel[columns_to_ffill_bfill] = (
    df_panel.groupby("conflict_id")[columns_to_ffill_bfill]
    .transform(lambda s: s.ffill().bfill())
)
df_panel

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,tjprire,tjrsym,tjrma,tjnr,imsrc,isocode_num,region_num,start_date,end_date,real_observation
0,205,1989-01-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,31.0,5.0,1990-04-01,2022-11-01,0
1,205,1989-02-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,31.0,5.0,1990-04-01,2022-11-01,0
2,205,1989-03-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,31.0,5.0,1990-04-01,2022-11-01,0
3,205,1989-04-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,31.0,5.0,1990-04-01,2022-11-01,0
4,205,1989-05-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,31.0,5.0,1990-04-01,2022-11-01,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,69.0,1.0,2024-12-01,2024-12-01,0
86828,16379,2024-09-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,69.0,1.0,2024-12-01,2024-12-01,0
86829,16379,2024-10-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,69.0,1.0,2024-12-01,2024-12-01,0
86830,16379,2024-11-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,69.0,1.0,2024-12-01,2024-12-01,0


### **World Bank Indicators**
---
<code>GDP per capita</code>, <code>GDP</code> from the World Bank will be used to normalize outcome variable and as control variables.

In [33]:
gdp_pc = pd.read_csv('../../data/input/gdp_pc/gdp_pc.csv')
gdp_pc["year_mo"] = pd.to_datetime(gdp_pc["year_mo"]).dt.to_period("M").dt.to_timestamp()
df_panel = df_panel.merge(gdp_pc, left_on=["isocode", "year_mo"], right_on=["isocode", "year_mo"], how="left")
df_panel

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,imsrc,isocode_num,region_num,start_date,end_date,real_observation,gdp_pc_current_usd,gdp_current_usd,population,log_gdp_pc_current_usd
0,205,1989-01-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,31.0,5.0,1990-04-01,2022-11-01,0,NaN,NaN,NaN,NaN
1,205,1989-02-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,31.0,5.0,1990-04-01,2022-11-01,0,NaN,NaN,NaN,NaN
2,205,1989-03-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,31.0,5.0,1990-04-01,2022-11-01,0,NaN,NaN,NaN,NaN
3,205,1989-04-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,31.0,5.0,1990-04-01,2022-11-01,0,NaN,NaN,NaN,NaN
4,205,1989-05-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,31.0,5.0,1990-04-01,2022-11-01,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,69.0,1.0,2024-12-01,2024-12-01,0,NaN,NaN,NaN,NaN
86828,16379,2024-09-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,69.0,1.0,2024-12-01,2024-12-01,0,NaN,NaN,NaN,NaN
86829,16379,2024-10-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,69.0,1.0,2024-12-01,2024-12-01,0,NaN,NaN,NaN,NaN
86830,16379,2024-11-01,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,69.0,1.0,2024-12-01,2024-12-01,0,NaN,NaN,NaN,NaN


### **Conflict Timing Features**
---

Convert dates to monthly periods, build a numeric month index, and core timing features: 

- <code>conflict_age</code>: the age of the conflict in months as the number of months between start_date and the current month
- <code>active_conflict_age</code>: the age of the conflict considering only months with fatalities reported
- <code>duration_months</code>: the number of months between start_date and end_date of each conflict (max end_date - min start_date)
- <code>active_duration_months</code>: the number of real observations (months with fatalities reported)

These features are used later as controls for running <code>csdid</code> with <code>first_agreement</code> as a treatment variable.


In [34]:
df_panel = df_panel.copy()

df_panel['year_mo'] = pd.to_datetime(df_panel['year_mo'], format='%Y-%m').dt.to_period('M')
df_panel['start_date'] = pd.to_datetime(df_panel['start_date'], format='%Y-%m').dt.to_period('M')
df_panel['end_date'] = pd.to_datetime(df_panel['end_date'], format='%Y-%m').dt.to_period('M')

base_date = df_panel['year_mo'].min()

#numeric month index starting from 1
df_panel['year_mo_numeric'] = (df_panel["year_mo"] - base_date).apply(lambda x: x.n+1)
df_panel['start_date_numeric'] = (df_panel['start_date'] - base_date).apply(lambda x: x.n+1)
df_panel['end_date_numeric'] = (df_panel['end_date'] - base_date).apply(lambda x: x.n+1)

#Conflict age (months since start) 
#-----------------------------------------------------------------------------------------------
df_panel["conflict_age"] = df_panel['year_mo_numeric']- df_panel['start_date_numeric']
#for the negative values in all the cases impute with missing
df_panel['conflict_age'] = df_panel['conflict_age'].apply(lambda x: x if x >=0 else np.nan)

age_thresholds = [6, 12, 18, 24, 30]
for months in age_thresholds:
    col = f'conflict_age_less_{months}m'
    df_panel[col] = np.where(
        df_panel['conflict_age'].notna(),
        (df_panel['conflict_age'] <= months).astype(int),
        np.nan,
    )

#Active Conflict Age (months with recorded violence)
#-----------------------------------------------------------------------------------------------
df_panel["active_conflict_age"] = (
    df_panel.groupby("conflict_id")["real_observation"].cumsum() - 1
)
df_panel['active_conflict_age'] = df_panel['active_conflict_age'].apply(lambda x: x if x >=0 else np.nan)

#Duration Conflict in Months
#-----------------------------------------------------------------------------------------------
#difference between end_date_numeric and start_date_numeric
df_panel['duration_months'] = (df_panel.groupby("conflict_id")["end_date_numeric"].transform("max") - df_panel.groupby("conflict_id")["start_date_numeric"].transform("min"))

# #Duration of Active Conflict in Months
#-----------------------------------------------------------------------------------------------
df_panel["active_duration_months"] = (
     df_panel.groupby("conflict_id")["real_observation"].transform(lambda x: x.sum())
     )

# Calendar features
df_panel['start_year'] = df_panel['start_date'].dt.year
df_panel['current_month'] = df_panel['year_mo'].dt.month

df_panel


,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,conflict_age_less_6m,conflict_age_less_12m,conflict_age_less_18m,conflict_age_less_24m,conflict_age_less_30m,active_conflict_age,duration_months,active_duration_months,start_year,current_month
0,205,1989-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,1
1,205,1989-02,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,2
2,205,1989-03,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,3
3,205,1989-04,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,4
4,205,1989-05,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2024,8
86828,16379,2024-09,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2024,9
86829,16379,2024-10,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2024,10
86830,16379,2024-11,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2024,11


### **First treatment date per conflict**
---

For each agreement type, compute the first month of treatment (<code>first\_{treatment}\_date</code>) and a conflict‑level treated indicator (<code>treated_{treatment}</code>). 

The first date is set to 0 when a conflict is never treated. These variables are used to define treatment timing in the event‑study and csdid models.


In [35]:
agreements = [
    "agreement", "comp_agreement", "subs_agreement", "cea_agreement",
    "cea_ceas_agreement", "cea_ceamix_agreement", "cea_rel_agreement"
]

g = df_panel.groupby("conflict_id", sort=False)
new_cols = {}

for col in agreements:
    # First treated month per conflict_id (NaN if never treated)
    first_month = (
        df_panel.loc[df_panel[col].eq(1)]
        .groupby("conflict_id")["year_mo_numeric"]
        .min()
    )
    # Map to all rows; 0 if never treated
    new_cols[f"first_{col}_date"] = (
        df_panel["conflict_id"].map(first_month).fillna(0).astype(int)
    )
    # Ever treated (0/1) per conflict_id
    new_cols[f"treated_{col}"] = g[col].transform("max").astype(int)

# Add all columns at once → no fragmentation
df_panel = pd.concat([df_panel, pd.DataFrame(new_cols)], axis=1)
df_panel


,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,first_subs_agreement_date,treated_subs_agreement,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement
0,205,1989-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0,0
1,205,1989-02,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0,0
2,205,1989-03,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0,0
3,205,1989-04,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0,0
4,205,1989-05,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0,0
86828,16379,2024-09,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0,0
86829,16379,2024-10,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0,0
86830,16379,2024-11,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0,0


### **Sequences**
---

In [36]:
def gen_since(series: pd.Series):
    series = (series == False)
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()
    result[groups == 0] = np.nan
    return result

df_panel['since_last_agreement'] = df_panel.groupby('conflict_id')['agreement'].transform(gen_since)

In [37]:
def gen_sequence_since_start(since_last_agreement, year_mo_numeric, first_agreement_date, sequence_size=12):
    """
    Generates cumulative sequence numbers that start counting
    only after the conflict's start_date.

    Parameters
    ----------
    since_last_agreement : pd.Series
        Series indicating months since last agreement.
    year_mo_numeric : pd.Series
        Numeric month-year index (1 for 1989-01, etc.).
    start_date : float or int
        Numeric start date of the conflict.
    sequence_size : int
        Threshold (in months) for incrementing sequence count.

    Returns
    -------
    pd.Series
        sequence indicator starting at 1 from the start_date.
        NaN before the conflict starts, and 0 if never treated.
    """

    # Create a copy to avoid modifying the original
    s = since_last_agreement.copy()
    seq = pd.Series(index=s.index, dtype="float")

    # Mask: only months after conflict start
    active_mask = year_mo_numeric >= first_agreement_date - sequence_size

    # Only compute sequence for months where conflict is active
    s_active = s[active_mask]

    if s_active.notna().sum() == 0:
        # No valid agreements after start_date → return NaN (filled with 0 later)
        seq.loc[:] = np.nan
        return seq

    # Detect when since_last_agreement crosses the threshold
    trigger = (s_active.shift(1) <= sequence_size) & (s_active == sequence_size + 1)

    # Cumulative sum + 1 = sequence number
    seq_active = trigger.cumsum() + 1
    seq_active = seq_active.fillna(1).astype(int)

    # Assign results
    seq.loc[active_mask] = seq_active

    # Before start_date, keep NaN (will be filled later with 0)
    return seq


In [38]:
df_panel = df_panel.sort_values(['conflict_id', 'year_mo_numeric'])

#Sequence for 12 months
#------------------------------------------------------------------------
df_panel['sequence_12m'] = np.nan  # initialize

treated = df_panel['treated_agreement'] == 1

for cid, group in df_panel.loc[treated].groupby('conflict_id'):
    start = group['first_agreement_date'].iloc[0]
    seq = gen_sequence_since_start(
        group['since_last_agreement'],
        group['year_mo_numeric'],
        start,
        sequence_size=12
    )
    df_panel.loc[group.index, 'sequence_12m'] = seq

# Fill 0 for untreated conflicts
df_panel.loc[~treated, 'sequence_12m'] = 0
df_panel['sequence_12m'] = df_panel['sequence_12m'].astype('Int64')  

In [39]:
df_panel.columns = df_panel.columns.str.replace(' ', '_').str.replace('-', '_').str.replace('.', '_').str.replace('|', '_')
df_panel['conflict_id'] = df_panel['conflict_id'].astype(int)
df_panel['best'] = df_panel['best'].astype(float)
df_panel['log_best'] = df_panel['log_best'].astype(float)
df_panel

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement,since_last_agreement,sequence_12m
0,205,1989-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,NaN,0
1,205,1989-02,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,NaN,0
2,205,1989-03,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,NaN,0
3,205,1989-04,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,NaN,0
4,205,1989-05,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,NaN,0
86828,16379,2024-09,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,NaN,0
86829,16379,2024-10,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,NaN,0
86830,16379,2024-11,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,NaN,0


### **Treatment variables**
---

In [40]:
# Define Conflict Active
#================================================================================

# Define any_violence: 1 if there were fatalities, 0 otherwise
df_panel['any_violence'] = (df_panel['log_best'] > 0).astype(int)

def gen_since(series: pd.Series):
    #count the number of consecutive months since last violence
    series = (series == False)
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()
    result[groups == 0] = np.nan
    return result

def gen_until(series: pd.Series) -> pd.Series:
    #count the number of consecutive months until next violence
    series = series[::-1]
    series = series == 0
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()[::-1]
    result[groups == 0] = np.nan
    return result

# Apply function within each conflict panel
df_panel['since_any_violence'] = df_panel.groupby('conflict_id')['any_violence'].transform(gen_since)
df_panel.year_mo = df_panel.year_mo.astype(str)
df_panel['year'] = df_panel['year_mo'].str[:4].astype(int)

# Define conflict_active according to the 24-month rule (and special 12-month rule for 1990)
df_panel['conflict_active'] = np.where(
    (df_panel['since_any_violence'] < 24) |
    ((df_panel['since_any_violence'] < 12) & (df_panel['year']<= 1990)),
    1, 0
)

#create agreements associated with conflict active
# ================================================================================

for i in ['agreement', 'comp_agreement', 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 
          'cea_rel_agreement', 'cea_ceamix_agreement', 'ce']:
    if i != 'ce':
        df_panel[f'{i}'] = np.where(
            (df_panel[f'{i}'] == 1) & (df_panel['conflict_active'] == 1),
            1, 0
        )
    else:
        df_panel[f'{i}asefire_mentions'] = np.where(
            (df_panel[f'{i}'] >= 1) & (df_panel['conflict_active'] == 1),
            1, 0)


df_panel['treated_ceasfire_mentions'] = (df_panel.groupby('conflict_id')['ceasefire_mentions'].transform('max'))

#Define definitive treatment variable
#--------------------------------------------------------------------------------
#combine cea_agreementes with ceasefire_mentions
df_panel['ceasfire_agreements_mentions'] = np.where(
    (df_panel['cea_agreement'] == 1) | (df_panel['ceasefire_mentions'] == 1),
    1,0
)
df_panel['treated_ceasfire_agreements_mentions'] = (df_panel.groupby('conflict_id')['ceasfire_agreements_mentions'].transform('max'))
df_panel

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,since_last_agreement,sequence_12m,any_violence,since_any_violence,year,conflict_active,ceasefire_mentions,treated_ceasfire_mentions,ceasfire_agreements_mentions,treated_ceasfire_agreements_mentions
0,205,1989-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,0,0,NaN,1989,0,0,0,0,0
1,205,1989-02,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,0,0,NaN,1989,0,0,0,0,0
2,205,1989-03,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,0,0,NaN,1989,0,0,0,0,0
3,205,1989-04,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,0,0,NaN,1989,0,0,0,0,0
4,205,1989-05,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,NaN,0,0,NaN,1989,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,0,0,NaN,2024,0,0,0,0,0
86828,16379,2024-09,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,0,0,NaN,2024,0,0,0,0,0
86829,16379,2024-10,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,0,0,NaN,2024,0,0,0,0,0
86830,16379,2024-11,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,NaN,0,0,NaN,2024,0,0,0,0,0


### **Treatment without ceasefire mentions**

In [41]:
# months to next ceasefire mention / agreement (por conflicto)
df_panel['until_ce_mentions'] = df_panel.groupby('conflict_id')['ceasefire_mentions'].transform(gen_until)
df_panel['ce_mentions_active'] = (df_panel['until_ce_mentions'] >= 6).astype(int)

df_panel['until_cea_agreement'] = df_panel.groupby('conflict_id')['cea_agreement'].transform(gen_until)
df_panel['cea_agreement_active'] = (df_panel['until_cea_agreement'] >= 6).astype(int)

active_6m = (df_panel['ce_mentions_active'].eq(1) & df_panel['cea_agreement_active'].eq(1))


# ============================================================
# Generate no ceasefire variables for agreement, comp_agreement, subs_agreement
# ============================================================

types = ['agreement', 'comp_agreement', 'subs_agreement']

for t in types:
    base = f'{t}_no_ceasefire'
    treated = f'treated_{t}s_no_ceasefire'
    var6m = f'{t}_no_ceasefire_mentions_agreement_6m'

    # t == 1 AND ce <= 0 AND cea_agreement == 0
    df_panel[base] = (
        df_panel[t].eq(1) &
        df_panel['ce'].le(0) &
        df_panel['cea_agreement'].eq(0)
    ).astype(int)

    # max per conflict
    df_panel[treated] = df_panel.groupby('conflict_id')[base].transform('max')

    # (active 6m) AND base
    df_panel[var6m] = (active_6m & df_panel[base].eq(1)).astype(int)

### **Define High and Low Intensity Conflicts**
---

In [42]:
# ----------------------------------------------------------------------
# 1️⃣ Calculate the fatalities pre agreement for treated conflicts
# ----------------------------------------------------------------------
fatalities_pre_agreement = (
    df_panel.loc[
        (df_panel['treated_agreement'] == 1) &
        (df_panel['year_mo_numeric'] < df_panel['first_agreement_date'])
    ]
    .groupby('conflict_id')['best']
    .sum()
    .rename('fatalities_pre_agreement')
)

# ----------------------------------------------------------------------
# 2️⃣ Merge with the main dataset (all the records of the same conflict get the same value)
# ----------------------------------------------------------------------
df_panel = df_panel.merge(fatalities_pre_agreement, on='conflict_id', how='left')

# ----------------------------------------------------------------------
# 3️⃣ Create a categorical variable based on the 1000 fatalities threshold
# ----------------------------------------------------------------------
df_panel['high_intensity'] = np.where(
    df_panel['fatalities_pre_agreement'] > 1000, 1,
    np.where(df_panel['fatalities_pre_agreement'].notna(), 0, np.nan)
)

# ----------------------------------------------------------------------
# 4️⃣ For control conflicts (without agreement): assign 0 for consistency
# ----------------------------------------------------------------------
df_panel.loc[df_panel['treated_agreement'] == 0, 'fatalities_pre_agreement'] = 0
df_panel.loc[df_panel['treated_agreement'] == 0, 'high_intensity'] = 0

# ----------------------------------------------------------------------
# 5️⃣ Quick check of how many treated conflicts are in each group
# ----------------------------------------------------------------------
summary = (
    df_panel.loc[df_panel['treated_agreement'] == 1]
    .drop_duplicates('conflict_id')
    .groupby('high_intensity')['conflict_id']
    .count()
)

print("Number of treated conflicts with more than 1000 pre-agreement fatalities:", summary.get(1, 0))
print("NNumber of treated conflicts with 1000 or fewer pre-agreement fatalities:", summary.get(0, 0))

Number of treated conflicts with more than 1000 pre-agreement fatalities: 29
NNumber of treated conflicts with 1000 or fewer pre-agreement fatalities: 42


In [43]:
df_panel

,conflict_id,year_mo,best,n_events,dyad_id,n_isocode,isocode_array,countries_array,type_of_violence,isocode,...,treated_agreements_no_ceasefire,agreement_no_ceasefire_mentions_agreement_6m,comp_agreement_no_ceasefire,treated_comp_agreements_no_ceasefire,comp_agreement_no_ceasefire_mentions_agreement_6m,subs_agreement_no_ceasefire,treated_subs_agreements_no_ceasefire,subs_agreement_no_ceasefire_mentions_agreement_6m,fatalities_pre_agreement,high_intensity
0,205,1989-01,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0.0,0.0
1,205,1989-02,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0.0,0.0
2,205,1989-03,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0.0,0.0
3,205,1989-04,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0.0,0.0
4,205,1989-05,0.0,0.0,NaN,0.0,[IRN],[Iran],1.0,IRN,...,0,0,0,0,0,0,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0.0,0.0
86828,16379,2024-09,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0.0,0.0
86829,16379,2024-10,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0.0,0.0
86830,16379,2024-11,0.0,0.0,NaN,0.0,[SOM],[Somalia],1.0,SOM,...,0,0,0,0,0,0,0,0,0.0,0.0


In [46]:
def count_factions(x):

    # 1️⃣ Si es lista o array → contar directamente
    if isinstance(x, (list, np.ndarray)):
        return len(x)

    # 2️⃣ Si es NaN → devolver NaN
    if pd.isna(x):
        return np.nan
    
    # 3️⃣ Si es string → separar por espacios y contar
    if isinstance(x, str):
        return len(x.split())
    
    # 4️⃣ Otros casos raros
    return np.nan

df_panel['n_factions'] = df_panel['dyad_id'].apply(count_factions)

df_panel[['conflict_id', 'year', 'dyad_id', 'n_factions']]

,conflict_id,year,dyad_id,n_factions
0,205,1989,NaN,NaN
1,205,1989,NaN,NaN
2,205,1989,NaN,NaN
3,205,1989,NaN,NaN
4,205,1989,NaN,NaN
...,...,...,...,...
86827,16379,2024,NaN,NaN
86828,16379,2024,NaN,NaN
86829,16379,2024,NaN,NaN
86830,16379,2024,NaN,NaN


In [47]:
df_panel.to_csv('../../data/output/conflict_level/conflict_panel.csv', index=False)